<a href="https://colab.research.google.com/github/Akshayes6/cricket-player-performance-predictor/blob/main/cricket_player_performance_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from datetime import datetime
from sklearn.preprocessing import LabelEncoder,StandardScaler,MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error,r2_score,mean_absolute_error,root_mean_squared_error

In [2]:
df = pd.read_csv('/content/Player_Performance_Dataset_Updated_Numeric.csv')
df1 = pd.read_csv('/content/updated_expanded_player_names_numeric_isout.csv')

In [3]:
df.shape
df1.shape

(1315, 12)

In [4]:
df.drop(['Match ID','player_id','country_id','fielders','Match Venue (Country)','Team1 Name','wicketType'],inplace=True,axis=1)

In [5]:
df.rename(columns={'team':'Country','player_name':'Player','Team2 Name':'Opponent','Match Venue (Stadium)':'stadium'},inplace=True)

In [6]:
new_order = ['Player','Country','Opponent','runs','balls','fours','sixes','strikeRate','isOut','Match Date','stadium']
df = df[new_order]
df.head()

,Player,Country,Opponent,runs,balls,fours,sixes,strikeRate,isOut,Match Date,stadium
0,Hamilton Masakadza,Zimbabwe,Pakistan,53.0,38.0,5.0,3.0,139.47,1.0,2008-10-12,Maple Leaf North-West Ground
1,Cephas Zhuwao,Zimbabwe,Pakistan,12.0,22.0,2.0,0.0,54.54,1.0,2008-10-12,Maple Leaf North-West Ground
2,Chamu Chibhabha,Zimbabwe,Pakistan,8.0,26.0,0.0,0.0,30.76,1.0,2008-10-12,Maple Leaf North-West Ground
3,Tatenda Taibu,Zimbabwe,Pakistan,4.0,6.0,1.0,0.0,66.66,1.0,2008-10-12,Maple Leaf North-West Ground
4,Salman Butt,Pakistan,Sri Lanka,44.0,41.0,4.0,1.0,107.31,1.0,2008-10-13,Maple Leaf North-West Ground


In [7]:
df1.head()

,player_name,runs,balls,fours,sixes,strikeRate,Inns,opposition,ground,match_date,country,isout
0,Shadab Khan,0.0,2.0,0.0,0.0,0.0,2,v Ireland,Lauderhill,2024-06-16,Pakistan,0
1,Tanzid Hasan,0.0,1.0,0.0,0.0,0.0,1,v Nepal,Kingstown,2024-06-16,Bangladesh,0
2,Pathum Nissanka,0.0,2.0,0.0,0.0,0.0,1,v Netherlands,Gros Islet,2024-06-16,Sri Lanka,0
3,Dasun Shanaka,0.0,1.0,0.0,0.0,0.0,1,v Netherlands,Gros Islet,2024-06-16,Sri Lanka,0
4,Finn Allen,0.0,2.0,0.0,0.0,0.0,2,v P.N.G.,Tarouba,2024-06-17,New Zealand,0


In [8]:
df1.drop('Inns',inplace=True,axis=1)

df1.rename(columns={'player_name':'Player','opposition':'Opponent','match_date':'Match Date','country':'Country','ground':'stadium','isout':'isOut'},inplace=True)

new_order1 = ['Player','Country','Opponent','runs','balls','fours','sixes','strikeRate','isOut','Match Date','stadium',]

df1['Opponent'] = df1['Opponent'].str.replace('v ','')
df1['isOut'] = df1['isOut'].replace({'True':1,'False':0})

df1 =df1[new_order1]
df1.head()

,Player,Country,Opponent,runs,balls,fours,sixes,strikeRate,isOut,Match Date,stadium
0,Shadab Khan,Pakistan,Ireland,0.0,2.0,0.0,0.0,0.0,0,2024-06-16,Lauderhill
1,Tanzid Hasan,Bangladesh,Nepal,0.0,1.0,0.0,0.0,0.0,0,2024-06-16,Kingstown
2,Pathum Nissanka,Sri Lanka,Netherlands,0.0,2.0,0.0,0.0,0.0,0,2024-06-16,Gros Islet
3,Dasun Shanaka,Sri Lanka,Netherlands,0.0,1.0,0.0,0.0,0.0,0,2024-06-16,Gros Islet
4,Finn Allen,New Zealand,P.N.G.,0.0,2.0,0.0,0.0,0.0,0,2024-06-17,Tarouba


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56357 entries, 0 to 56356
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Player      55818 non-null  object 
 1   Country     56331 non-null  object 
 2   Opponent    56357 non-null  object 
 3   runs        41872 non-null  float64
 4   balls       41872 non-null  float64
 5   fours       41872 non-null  float64
 6   sixes       41872 non-null  float64
 7   strikeRate  41872 non-null  float64
 8   isOut       56331 non-null  float64
 9   Match Date  56115 non-null  object 
 10  stadium     56357 non-null  object 
dtypes: float64(6), object(5)
memory usage: 4.7+ MB


In [10]:
df.isna().sum()

,0
Player,539
Country,26
Opponent,0
runs,14485
balls,14485
fours,14485
sixes,14485
strikeRate,14485
isOut,26
Match Date,242


In [11]:
# These are core batting stats. Missing values probably indicate players who did not bat in the innings

for i in ['runs','balls','fours','sixes','strikeRate']:
  df[i].fillna(0,inplace=True)

for j in ['Country','Player']:
  df[j].fillna('Unknown',inplace=True)


df['Match Date'].fillna('Unknown Date',inplace=True)

df['stadium'].fillna('Unknown Stadium',inplace=True)

df['isOut'].fillna(0,inplace=True)

df['Opponent'].fillna('Unknown Opponent',inplace=True)

/tmp/ipykernel_507/1557480281.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[i].fillna(0,inplace=True)
/tmp/ipykernel_507/1557480281.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col:

In [12]:
df = df.astype({'runs':int,
           'balls':int,
           'fours':int,
           'sixes':int,
           'isOut':int
           })
df['Match Date'] = pd.to_datetime(df['Match Date'],errors='coerce')
df.dtypes

,0
Player,object
Country,object
Opponent,object
runs,int64
balls,int64
fours,int64
sixes,int64
strikeRate,float64
isOut,int64
Match Date,datetime64[ns]


In [13]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1315 entries, 0 to 1314
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Player      1315 non-null   object 
 1   Country     1315 non-null   object 
 2   Opponent    1315 non-null   object 
 3   runs        1036 non-null   float64
 4   balls       1315 non-null   float64
 5   fours       1315 non-null   float64
 6   sixes       1315 non-null   float64
 7   strikeRate  1302 non-null   float64
 8   isOut       1315 non-null   int64  
 9   Match Date  1315 non-null   object 
 10  stadium     1315 non-null   object 
dtypes: float64(5), int64(1), object(5)
memory usage: 113.1+ KB


In [14]:
df1['Opponent'].unique()

array(['Ireland', 'Nepal', 'Netherlands', 'P.N.G.', 'West Indies',
       'U.S.A.', 'India', 'Australia', 'England', 'Afghanistan',
       'South Africa', 'Zimbabwe', 'Sri Lanka', 'Scotland', 'Bangladesh',
       'New Zealand', 'Pakistan', 'Uganda', 'Oman', 'Namibia', 'Canada'],
      dtype=object)

In [15]:
df1.isna().sum()


,0
Player,0
Country,0
Opponent,0
runs,279
balls,0
fours,0
sixes,0
strikeRate,13
isOut,0
Match Date,0


In [16]:
for i in ['runs','strikeRate']:
  df1[i].fillna(0,inplace=True)

/tmp/ipykernel_507/783071448.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df1[i].fillna(0,inplace=True)


In [17]:
df1.isna().sum()

,0
Player,0
Country,0
Opponent,0
runs,0
balls,0
fours,0
sixes,0
strikeRate,0
isOut,0
Match Date,0


In [18]:
df1 = df1.astype({'runs':int,
           'balls':int,
           'fours':int,
           'sixes':int,
           })
df1['Match Date'] = pd.to_datetime(df1['Match Date'],errors='coerce')
df1.dtypes

,0
Player,object
Country,object
Opponent,object
runs,int64
balls,int64
fours,int64
sixes,int64
strikeRate,float64
isOut,int64
Match Date,datetime64[ns]


In [19]:
df.duplicated().sum()

np.int64(229)

In [20]:
df.drop_duplicates(inplace=True)

In [21]:
df1.duplicated().sum()

np.int64(0)

In [22]:
new_df = pd.concat([df,df1],ignore_index=True)
new_df

,Player,Country,Opponent,runs,balls,fours,sixes,strikeRate,isOut,Match Date,stadium
0,Hamilton Masakadza,Zimbabwe,Pakistan,53,38,5,3,139.47,1,2008-10-12,Maple Leaf North-West Ground
1,Cephas Zhuwao,Zimbabwe,Pakistan,12,22,2,0,54.54,1,2008-10-12,Maple Leaf North-West Ground
2,Chamu Chibhabha,Zimbabwe,Pakistan,8,26,0,0,30.76,1,2008-10-12,Maple Leaf North-West Ground
3,Tatenda Taibu,Zimbabwe,Pakistan,4,6,1,0,66.66,1,2008-10-12,Maple Leaf North-West Ground
4,Salman Butt,Pakistan,Sri Lanka,44,41,4,1,107.31,1,2008-10-13,Maple Leaf North-West Ground
...,...,...,...,...,...,...,...,...,...,...,...
57438,JP Inglis,Australia,England,37,27,2,1,137.03,0,2024-09-11,Southampton
57439,Liam Livingstone,England,Australia,37,27,4,1,137.03,0,2024-09-11,Southampton
57440,R Powell,West Indies,Sri Lanka,37,27,1,3,137.03,0,2024-10-17,Dambulla
57441,Pathum Nissanka,Sri Lanka,New Zealand,37,28,3,2,132.14,0,2024-12-30,Mount Maunganui


In [23]:
new_df.to_csv('player batting data.csv', index=False)

In [24]:
df_past_10 = new_df[new_df['Match Date'].dt.year >= (pd.Timestamp.now().year - 10)]

In [25]:
top_teams = ['India', 'England', 'Australia', 'New Zealand', 'Pakistan',
             'South Africa', 'Sri Lanka', 'West Indies', 'Bangladesh', 'Afghanistan']

df_main = df_past_10[df_past_10['Country'].isin(top_teams)]

In [26]:
df_main.reset_index(drop=True,inplace=True)
df_main.sort_values(by=['Player','Match Date'],inplace=True)

/tmp/ipykernel_507/4080674498.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main.sort_values(by=['Player','Match Date'],inplace=True)


In [27]:
df_main.head()
df_main.shape

(14234, 11)

In [28]:
df_main = df_main[~((df_main['strikeRate'] > 200) & (df_main['balls'] < 10))]

In [29]:

df_main.shape

(13838, 11)

In [30]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13838 entries, 14202 to 13075
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Player      13838 non-null  object        
 1   Country     13838 non-null  object        
 2   Opponent    13838 non-null  object        
 3   runs        13838 non-null  int64         
 4   balls       13838 non-null  int64         
 5   fours       13838 non-null  int64         
 6   sixes       13838 non-null  int64         
 7   strikeRate  13838 non-null  float64       
 8   isOut       13838 non-null  int64         
 9   Match Date  13838 non-null  datetime64[ns]
 10  stadium     13838 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(5), object(4)
memory usage: 1.3+ MB


In [31]:
df_main.describe()


,runs,balls,fours,sixes,strikeRate,isOut,Match Date
count,13838.000000,13838.000000,13838.000000,13838.000000,13838.000000,13838.000000,13838
mean,13.356988,10.649516,1.162162,0.558968,75.534945,0.517994,2020-12-21 14:35:46.784217088
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2016-01-07 00:00:00
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2018-10-22 00:00:00
50%,4.000000,6.000000,0.000000,0.000000,75.000000,1.000000,2021-08-06 00:00:00
75%,19.000000,16.000000,2.000000,1.000000,130.760000,1.000000,2022-11-04 00:00:00
max,172.000000,76.000000,16.000000,16.000000,357.140000,1.000000,2025-02-02 00:00:00
std,19.686806,12.964147,1.951091,1.231978,69.335067,0.499694,NaN


**EDA**

---



In [32]:
fig = px.histogram(df_main,'runs',nbins=30,title='Distribution of Runs',labels={'runs':'Run Scored'},
                   marginal='box')
fig.update_layout(bargap=0.1)
fig.show()

In [33]:
fig = px.scatter(df_main,x='balls',y='runs',hover_data=['Player','Country'],size='balls',color='Country',
                     title='Scatter Plot of Runs vs Balls',labels={'runs':'Runs Scored','balls':'Balls'})
fig.update_traces(marker=dict(line=dict(width=.5, color='DarkSlateGrey')))
fig.update_layout(title_x=0.45)
fig.show()

In [34]:
fig = px.scatter(df_main,x='strikeRate',y='runs',title='Scatter Plot of Runs Scored vs Strike rate',
                labels={'runs':'Runs Scored','strikeRate':'Strike Rate'},hover_data=['Player'],color='balls')
fig.update_traces(marker=dict(line=dict(width=.5)))
fig.update_layout(title_x=0.45)
fig.show()

In [35]:
runs_per_year = df_main.groupby(df_main['Match Date'].dt.year) ['runs'].sum().reset_index()

fig = px.line(runs_per_year,x='Match Date',y='runs',labels={'runs':'Total Runs','Match Date':'Years'})
fig.update_traces(line=dict(width=5))
fig.update_xaxes(dtick='1')
fig.show()

In [36]:
top_team = df_main.groupby('Country')['runs'].sum().sort_values(ascending=True).reset_index()

fig = px.bar(top_team,x='Country',y='runs',color='runs',labels={'runs':'Total Runs'},
             title='Total Run Scoring Countries')
fig.update_layout(title_x=0.45)
fig.show()

In [37]:
top_players = df_main.groupby(['Player','Country']) ['runs'].sum().sort_values(ascending=False).head(10).reset_index()

fig = px.bar(top_players,'Player','runs',hover_data=['Country'],color='Country',title='Top Run Scorers',
             text_auto=True,height=600)
fig.update_layout(title_x=.45,xaxis_tickangle=0)
fig.show()

In [38]:
top_runs_inn=df_main.groupby(['Player','Country','Opponent'])['runs'].max().sort_values(ascending=False).head(10).reset_index()

fig = px.bar(top_runs_inn,'Player','runs',hover_data=['Country','Opponent'],color='runs',
             text_auto=True,title='Most runs in an innings in T20Is (2015-2025)')
fig.update_layout(title_x=.45)
fig.show()

In [39]:
sixes_fours = df_main.groupby(df_main['Match Date'].dt.year) [['sixes','fours']].sum().reset_index()

fig = px.bar(sixes_fours,x='Match Date',y=['fours','sixes'],barmode='group',title=
             'Total Sixes and Fours Scored Over the Years',labels={'Match Date':'Year','value':'Count'})
fig.update_layout(title_x=0.45)
fig.update_xaxes(dtick='1')
fig.show()

In [40]:
top_six_scorers=df_main.groupby(['Player','Country'])['sixes'].sum().sort_values(ascending=False).head(10).reset_index()

fig = px.bar(top_six_scorers,'Player','sixes',color='Country',title='Top 10 Six Hitters',
                              labels={'sixes':'Total Sixes','balls':'Total Balls Faced'})
fig.update_layout(title_x=0.5)

fig.show()

In [41]:
top_four_scorers=df_main.groupby(['Player','Country'])['fours'].sum().sort_values(ascending=False).head(10).reset_index()

fig = px.bar(top_four_scorers,'Player','fours',color='Country',title='Top 10 Six Hitters',
                              labels={'sixes':'Total Sixes','balls':'Total Balls Faced'})
fig.update_layout(title_x=0.5)
fig.show()

In [42]:
total_outs = df_main.loc[df_main['isOut']==1].groupby('Player') ['isOut'].count().reset_index()
total_runs = df_main.groupby('Player')['runs'].sum().reset_index()

df_player_avg=pd.merge(total_runs,total_outs,on='Player',how='inner')
df_player_avg['Avg'] = df_player_avg['runs']/df_player_avg['isOut']

best_avgs = df_player_avg.loc[(df_player_avg['runs'] > 1000) & (df_player_avg['Avg'] > 30)].sort_values(ascending=False,by='Avg')

In [43]:
fig = px.bar(best_avgs,'Player','Avg',title='Players with Highest Average (Runs > 1000)',
             labels={'Avg': 'Average'},color='Avg',)
fig.update_layout(xaxis_tickangle=-45,title_x=.45)
fig.show()

In [44]:
heat_map = df_main.drop(columns='isOut').select_dtypes(include=['number']).corr()
heat_map = df_main.drop(columns='isOut').select_dtypes(include=['number']).corr()

fig = px.imshow(heat_map,x=heat_map.columns,y=heat_map.index,text_auto=True,title='Correlation Heatmap',
                  aspect=1)
fig.update_layout(title_x=0.45)
fig.show()

In [45]:
df_main.sort_values(by=['Player','Opponent','Match Date'],inplace=True)
# df_main[(df_main['Player']=='Virat Kohli') & (df_main['Opponent']=='Australia')]

In [46]:
# df_main['Total Runs'] = df_main.groupby(['Player','Opponent']) ['runs'].transform('sum')

# out=df_main.loc[df_main['isOut']==1].groupby(['Player','Opponent']) ['isOut'].transform('count')
# out

df_main[(df_main['Player']=='Virat Kohli') & (df_main['Opponent']=='Australia')]

,Player,Country,Opponent,runs,balls,fours,sixes,strikeRate,isOut,Match Date,stadium
112,Virat Kohli,India,Australia,90,55,9,2,163.63,0,2016-01-26,Adelaide Oval
134,Virat Kohli,India,Australia,59,33,7,1,178.78,0,2016-01-29,Melbourne Cricket Ground
2538,Virat Kohli,India,Australia,0,2,0,0,0.00,1,2017-10-10,Barsapara Cricket Stadium
4275,Virat Kohli,India,Australia,24,17,3,0,141.17,1,2019-02-24,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...
4297,Virat Kohli,India,Australia,72,38,2,6,189.47,0,2019-02-27,M Chinnaswamy Stadium
5763,Virat Kohli,India,Australia,9,9,1,0,100.00,1,2020-12-04,Manuka Oval
10765,Virat Kohli,India,Australia,2,7,0,0,28.57,1,2022-09-20,Punjab Cricket Association IS Bindra Stadium
12936,Virat Kohli,India,Australia,0,5,0,0,0.00,0,2024-06-24,Gros Islet


**Feature** **Engineering**

---



In [47]:
# df_main.groupby(['Player','Opponent']).agg(total=('runs','sum'),totalout=('isOut','count'))

In [48]:
def recent_performance(data, n=5):

    data = data.sort_values(by=['Player','Opponent','Match Date'])

    data['Match_rank'] =data.groupby(['Player','Opponent']).cumcount(ascending=False)

    recent_matches = data[data['Match_rank'] < 5]

    summary = recent_matches.groupby(['Player', 'Opponent']).agg(
        total_runs=('runs','sum'),
        total_dismissals=('isOut','count'),
        total_balls=('balls','sum')
    ).reset_index()

    summary['recent_avg_runs_against_opponent'] = summary.apply(lambda row: row['total_runs']/row['total_dismissals'] if row['total_dismissals']>0 else row['total_runs'],axis=1)

    summary['recent_strike_rate_against_opponent'] = summary.apply(lambda row: (row['total_runs']/row['total_balls'])*100 if row['total_balls']>0 else 0,axis=1)

    return summary[['Player', 'Opponent', 'recent_avg_runs_against_opponent', 'recent_strike_rate_against_opponent']]


In [49]:
recent_stats_df = recent_performance(df_main,n=5)
# recent_stats_df[(recent_stats_df['Player']=='Virat Kohli') & (recent_stats_df['Opponent']=='Australia')]

In [50]:
def career_stats(data):

    career_summary = data.groupby('Player').agg(
        total_runs=('runs', 'sum'),
        total_dismissals=('isOut', 'sum'),
        total_balls=('balls', 'sum')
    ).reset_index()

    career_summary['career_avg_runs'] = career_summary.apply(
        lambda row: row['total_runs'] / row['total_dismissals'] if row['total_dismissals'] > 0 else row['total_runs'],
        axis=1
    )
    career_summary['career_strike_rate'] = career_summary.apply(
        lambda row: (row['total_runs'] / row['total_balls']) * 100 if row['total_balls'] > 0 else 0,
        axis=1
    )

    opponent_summary = data.groupby(['Player', 'Opponent']).agg(
        career_runs_against_opponent=('runs', 'sum'),
        total_dismissals_against_opponent=('isOut', 'sum'),
        total_balls_against_opponent=('balls', 'sum')
    ).reset_index()

    opponent_summary['career_avg_runs_against_opponent'] = opponent_summary.apply(
        lambda row: row['career_runs_against_opponent'] / row['total_dismissals_against_opponent']
        if row['total_dismissals_against_opponent'] > 0 else row['career_runs_against_opponent'],
        axis=1
    )
    opponent_summary['career_strike_rate_against_opponent'] = opponent_summary.apply(
        lambda row: (row['career_runs_against_opponent'] / row['total_balls_against_opponent']) * 100
        if row['total_balls_against_opponent'] > 0 else 0,
        axis=1
    )


    final_df = opponent_summary.merge(
        career_summary[['Player', 'career_avg_runs', 'career_strike_rate']],
        on='Player', how='left'
    )

    return final_df[['Player', 'Opponent', 'career_avg_runs', 'career_strike_rate',
                     'career_runs_against_opponent', 'career_avg_runs_against_opponent',
                     'career_strike_rate_against_opponent']]


In [51]:
career_stats_df = career_stats(df_main)

In [52]:
feature_df = pd.merge(recent_stats_df,career_stats_df,on=['Player','Opponent'])
feature_df.head()

,Player,Opponent,recent_avg_runs_against_opponent,recent_strike_rate_against_opponent,career_avg_runs,career_strike_rate,career_runs_against_opponent,career_avg_runs_against_opponent,career_strike_rate_against_opponent
0,A Athanaze,South Africa,23.0,127.777778,69.000000,127.777778,69,69.0,127.777778
1,A Simelane,India,5.0,78.947368,16.000000,69.565217,15,15.0,78.947368
2,A Simelane,Pakistan,1.0,25.000000,16.000000,69.565217,1,1.0,25.000000
3,AAP Atkinson,India,2.0,15.384615,2.000000,15.384615,2,2.0,15.384615
4,AB de Villiers,Afghanistan,64.0,220.689655,38.846154,162.903226,64,64.0,220.689655


In [53]:
df_model = pd.merge(df_main,feature_df,on=['Player','Opponent'],how='inner')


In [54]:
# df_model = df_model[df_model['runs']>0]

In [55]:
# venue_stats = df_main.groupby(['Player', 'stadium']).agg({'runs': 'mean', 'strikeRate': 'mean'}).reset_index()
# venue_stats.rename(columns={'runs': 'avg_runs_at_venue', 'strikeRate': 'avg_strike_rate_at_venue'}, inplace=True)
# df_model = pd.merge(df_model, venue_stats, on=['Player', 'stadium'], how='left')

In [56]:
# df_model = df_model[df_model['runs']>0]
df_model.shape


(13838, 18)

In [57]:
X = df_model.drop(columns=['Country','runs', 'Match Date', 'stadium','Player', 'Opponent','strikeRate'])
y = df_model['runs']


In [58]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [59]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [60]:
def evaluate_models(models,X_train,y_train,X_test,y_test):
  results = {}

  for name, model in models.items():
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test,y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test,y_pred)

    results[name] = {'MAE':mae, 'MAPE':mape,'R2':r2}

  best_model = min(results, key=lambda x:results[x]['MAE'])
  print(f"Best Performing Model: {best_model}\n")

  for model,metrics in results.items():
      print(f"{model} Performance:")
      print(f"MAE : {metrics['MAE']: .4f}")
      print(f"MAPE : {metrics['MAPE']: .4f}")
      print(f"R2 : {metrics['R2']: .4f}\n")

  return best_model,results

models = {
    'Linear Regresssion':LinearRegression(),
    'Decision Tree':DecisionTreeRegressor(),
    'Random Forest':RandomForestRegressor(),
    'XGBoost':XGBRegressor()
}


In [61]:
best_model = evaluate_models(models,X_train,y_train,X_test,y_test)

Best Performing Model: Random Forest

Linear Regresssion Performance:
MAE :  2.0980
MAPE :  2998688913273574.0000
R2 :  0.9497

Decision Tree Performance:
MAE :  2.2222
MAPE :  861779841990814.0000
R2 :  0.9378

Random Forest Performance:
MAE :  1.5745
MAPE :  909109734069766.5000
R2 :  0.9718

XGBoost Performance:
MAE :  1.5957
MAPE :  975629000376320.0000
R2 :  0.9702



In [62]:
model = RandomForestRegressor()
model.fit(X_train,y_train)

RandomForestRegressor()

In [63]:
def predict_runs(player_name, opponent):

  input_data = df_model[(df_model['Player']==player_name) & (df_model['Opponent']==opponent)]

  if input_data.empty:
    return f"No data available for player '{player_name}' against opponent '{opponent}'."


  input_data = input_data[['balls','fours', 'sixes','isOut', 'recent_avg_runs_against_opponent',
       'recent_strike_rate_against_opponent', 'career_avg_runs',
       'career_strike_rate', 'career_runs_against_opponent',
       'career_avg_runs_against_opponent', 'career_strike_rate_against_opponent']].mean().to_frame().T


  input_data = scaler.transform(input_data)

  predicted_runs = model.predict(input_data)


  return int(round(predicted_runs[0]))

In [64]:
# print(predict_runs('Virat Kohli','Australia'))

In [65]:
player_name = "Virat Kohli"
opponent = "Australia"

predicted = predict_runs(player_name, opponent)
print(f"Predicted Runs for {player_name} against {opponent}: {predicted}")

Predicted Runs for Virat Kohli against Australia: 30
